<a href="https://colab.research.google.com/github/smerarawal/IIT-BHU-Proj/blob/main/ucophyrehab_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

print("Click 'Choose Files' below to upload your JSON file:")
uploaded = files.upload()

# This code renames whatever you uploaded to exactly what our pipeline expects
for filename in uploaded.keys():
    os.rename(filename, "uploaded_exercise_data.json")
    print(f"\nSuccessfully uploaded and renamed to: uploaded_exercise_data.json")

Click 'Choose Files' below to upload your JSON file:


Saving dataset_3d_with_angles.json to dataset_3d_with_angles.json

🎉 Successfully uploaded and renamed to: uploaded_exercise_data.json


In [25]:
import os
import json
import numpy as np
import pandas as pd

def calculate_3d_vector_angle(v1, v2):
    dot_prod = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    cos_ang = dot_prod / (norm_v1 * norm_v2 + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_ang, -1.0, 1.0)))

def extract_features_from_colab_json_final(json_path):
    if not os.path.exists(json_path):
        print(f"Error: File not found at path '{json_path}'")
        return pd.DataFrame()

    with open(json_path, 'r') as f:
        try:
            payload = json.load(f)
        except Exception as e:
            print(f"Error parsing JSON structure: {e}")
            return pd.DataFrame()

    records = payload.get("data", [payload])
    print(f"Processing {len(records)} movement sequences...")
    extracted_rows = []

    for idx, record in enumerate(records):
        folder_id = record.get("folder", "0")
        ex_id = record.get("exercise", "01")
        side = record.get("side", "left")
        position = record.get("position", "seated")

        exercise_label = f"uco_{position}_ex{ex_id}_{side}"
        rep_id = f"colab_f{folder_id}_ex{ex_id}_{side}"
        subject_id = f"colab_s{folder_id}"

        frames_list = record.get("frames", [])
        if not frames_list:
            continue

        knee_angles = []
        hip_angles = []
        ankle_angles = []
        valgus_deviations = []

        for frame in frames_list:
            joints = frame.get("joints", {})

            p_ang = None
            if "angle" in joints and joints["angle"] is not None:
                p_ang = float(joints["angle"])
            elif "angle" in frame and frame["angle"] is not None:
                p_ang = float(frame["angle"])

            h_ang = 180.0
            a_ang = 90.0

            try:
                h_data = joints.get("l_hip" if side == "left" else "r_hip") or joints.get("l_hip") or joints.get("hip")
                k_data = joints.get("l_knee" if side == "left" else "r_knee") or joints.get("l_knee") or joints.get("knee")
                a_data = joints.get("l_ankle" if side == "left" else "r_ankle") or joints.get("l_ankle") or joints.get("ankle")

                if h_data and k_data and a_data:
                    hip = np.array([float(h_data["x"]), float(h_data["y"]), float(h_data["z"])])
                    knee = np.array([float(k_data["x"]), float(k_data["y"]), float(k_data["z"])])
                    ankle = np.array([float(a_data["x"]), float(a_data["y"]), float(a_data["z"])])

                    if p_ang is None:
                        v_knee_hip = hip - knee
                        v_knee_ankle = ankle - knee
                        p_ang = calculate_3d_vector_angle(v_knee_hip, v_knee_ankle)

                    v_hip_knee = knee - hip
                    vertical_axis = np.array([0.0, -1.0, 0.0])
                    h_ang = calculate_3d_vector_angle(v_hip_knee, vertical_axis)

                    v_ankle_knee = knee - ankle
                    horizontal_axis = np.array([1.0, 0.0, 0.0])
                    a_ang = calculate_3d_vector_angle(v_ankle_knee, horizontal_axis)

                    v_hk_2d = knee[:2] - hip[:2]
                    v_ha_2d = ankle[:2] - hip[:2]
                    valgus_deviations.append(calculate_3d_vector_angle(v_hk_2d, v_ha_2d))
            except Exception:
                pass

            if p_ang is not None:
                knee_angles.append(p_ang)
                hip_angles.append(h_ang)
                ankle_angles.append(a_ang)

        if not knee_angles:
            continue

        prim_traj = np.array(knee_angles)
        hip_traj = np.array(hip_angles)
        ank_traj = np.array(ankle_angles)

        angles_dict = {
            'left_hip':    hip_traj if side == "left" else np.ones_like(prim_traj) * 180.0,
            'left_knee':   prim_traj if side == "left" else np.ones_like(prim_traj) * 180.0,
            'left_ankle':  ank_traj if side == "left" else np.ones_like(prim_traj) * 90.0,
            'right_hip':   hip_traj if side == "right" else np.ones_like(prim_traj) * 180.0,
            'right_knee':  prim_traj if side == "right" else np.ones_like(prim_traj) * 180.0,
            'right_ankle': ank_traj if side == "right" else np.ones_like(prim_traj) * 90.0
        }

        peak_valgus = np.max(np.abs(np.array(valgus_deviations) - valgus_deviations[0])) if valgus_deviations else 0.0

        try:
            feats = extract_all_features(
                angles=angles_dict,
                exercise_label=exercise_label,
                valgus_angle=float(peak_valgus),
                rep_id=rep_id
            )
            feats['target_label'] = 1
            feats['subject_id'] = subject_id
            extracted_rows.append(feats)
        except Exception as e:
            print(f"Feature matrix mapping failed for {rep_id}: {str(e)}")

    return pd.DataFrame(extracted_rows)

# Update with your original raw JSON file path string
uploaded_json_path = "/content/uploaded_exercise_data.json"
output_csv_path = "corrected_json_features_3.csv"

df_new_features = extract_features_from_colab_json_final(uploaded_json_path)

if not df_new_features.empty:
    df_new_features.to_csv(output_csv_path, index=False)
    print(f"Feature matrix generated successfully. Corrected Dimensions: {df_new_features.shape}")
else:
    print("Script finished. Output matrix remains empty.")

Processing 432 movement sequences...


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Feature matrix generated successfully. Corrected Dimensions: (395, 142)
